In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/sjadhav26/cotton-data/Cotton-Original-Augmented/curl virus/mild/20241029_133531.jpg
/kaggle/input/datasets/sjadhav26/cotton-data/Cotton-Original-Augmented/curl virus/mild/20241029_142421.jpg
/kaggle/input/datasets/sjadhav26/cotton-data/Cotton-Original-Augmented/curl virus/mild/20241029_135328.jpg
/kaggle/input/datasets/sjadhav26/cotton-data/Cotton-Original-Augmented/curl virus/mild/20241029_145723.jpg
/kaggle/input/datasets/sjadhav26/cotton-data/Cotton-Original-Augmented/curl virus/mild/20241029_153400.jpg
/kaggle/input/datasets/sjadhav26/cotton-data/Cotton-Original-Augmented/curl virus/mild/20241029_141100.jpg
/kaggle/input/datasets/sjadhav26/cotton-data/Cotton-Original-Augmented/curl virus/mild/20241029_142230.jpg
/kaggle/input/datasets/sjadhav26/cotton-data/Cotton-Original-Augmented/curl virus/mild/20241029_140314.jpg
/kaggle/input/datasets/sjadhav26/cotton-data/Cotton-Original-Augmented/curl virus/mild/20241029_153600.jpg
/kaggle/input/datasets/sjadhav26/cott

In [2]:
# Install required packages
!pip install -q timm wandb albumentations

# Core imports
import os
import cv2
import numpy as np
from PIL import Image
import torch
import torch.nn as nn
import timm
from torch.optim.lr_scheduler import CosineAnnealingLR
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix
from torch.utils.data import Dataset, DataLoader
import albumentations as A
from albumentations.pytorch import ToTensorV2
import wandb

# Initialize your WandB tracking under the FarmGuard project
wandb.init(project="FarmGuard", name="densenet121-full-production-30eps")

# Set up device and directory structure
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
os.makedirs("/kaggle/working/weights", exist_ok=True)

print(f"Environment ready. Running on device: {device}")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

  2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

  ········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: sgjadhav17 (sgjadhav17-iiit-nagpur-official) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


Environment ready. Running on device: cuda


In [3]:
dataset_dir = '/kaggle/input/datasets/sjadhav26/cotton-data/Cotton-Original-Augmented'

# 2. Build class-to-index mapping dictionary securely
class_names = sorted([d for d in os.listdir(dataset_dir) if os.path.isdir(os.path.join(dataset_dir, d))])
class_to_idx = {cls_name: i for i, cls_name in enumerate(class_names)}

# 3. Gather raw file paths and labels
raw_image_paths = []
raw_labels = []

for class_name in class_names:
    class_path = os.path.join(dataset_dir, class_name)
    for root, _, files in os.walk(class_path):
        for img_name in files:
            if img_name.lower().endswith(('jpg', 'jpeg', 'png', 'bmp', 'webp')):
                raw_image_paths.append(os.path.join(root, img_name))
                raw_labels.append(class_to_idx[class_name])

# 4. Data Cleaning and Integrity Check Pass
clean_image_paths = []
clean_labels = []
corrupted_count = 0

print("Running data cleaning and verification scan...")
for path, label in zip(raw_image_paths, raw_labels):
    try:
        with Image.open(path) as img:
            img.verify()
        
        img_cv = cv2.imread(path)
        if img_cv is None:
            corrupted_count += 1
            continue
            
        clean_image_paths.append(path)
        clean_labels.append(label)
    except Exception:
        corrupted_count += 1

print(f"Scan Complete.")
print(f" - Valid images ready for training: {len(clean_image_paths)}")
print(f" - Corrupted/skipped files removed: {corrupted_count}")

# 5. Strict Stratified Split (75% Train, 10% Validation, 15% Test) with Fixed Seed (42)
train_val_paths, test_paths, train_val_labels, test_labels = train_test_split(
    clean_image_paths, clean_labels, test_size=0.15, random_state=42, stratify=clean_labels
)

train_paths, val_paths, train_labels, val_labels = train_test_split(
    train_val_paths, train_val_labels, test_size=0.1176, random_state=42, stratify=train_val_labels
)

# 6. Define Field-Realistic Albumentations Pipelines
train_transform = A.Compose([
    A.Resize(224, 224),
    A.MotionBlur(p=0.2),
    A.RandomShadow(p=0.2),
    A.Perspective(p=0.2),
    A.CoarseDropout(max_holes=8, max_height=24, max_width=24, p=0.3),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

val_test_transform = A.Compose([
    A.Resize(224, 224),
    A.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
    ToTensorV2()
])

# 7. Custom PyTorch Dataset Class
class CottonDataset(Dataset):
    def __init__(self, paths, labels, transform=None):
        self.paths = paths
        self.labels = labels
        self.transform = transform

    def __len__(self):
        return len(self.paths)

    def __getitem__(self, idx):
        image = cv2.imread(self.paths[idx])
        image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
        label = self.labels[idx]

        if self.transform:
            augmented = self.transform(image=image)
            image = augmented['image']

        return image, label

# 8. Instantiate Datasets and DataLoaders
train_dataset = CottonDataset(train_paths, train_labels, transform=train_transform)
val_dataset = CottonDataset(val_paths, val_labels, transform=val_test_transform)
test_dataset = CottonDataset(test_paths, test_labels, transform=val_test_transform)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False, num_workers=2, pin_memory=True)

print("FarmGuard DataLoaders successfully created and verified!")

Running data cleaning and verification scan...
Scan Complete.
 - Valid images ready for training: 3568
 - Corrupted/skipped files removed: 0
FarmGuard DataLoaders successfully created and verified!


/tmp/ipykernel_58/598905145.py:59: UserWarning: Argument(s) 'max_holes, max_height, max_width' are not valid for transform CoarseDropout
  A.CoarseDropout(max_holes=8, max_height=24, max_width=24, p=0.3),


In [8]:
from torch.optim.lr_scheduler import CosineAnnealingLR
model = timm.create_model('densenet121', pretrained=True, num_classes=4)
model.to(device)

criterion = nn.CrossEntropyLoss()
# Conservative learning rate with weight decay for stable fine-tuning
optimizer = torch.optim.AdamW(model.parameters(), lr=5e-5, weight_decay=1e-2)

epochs = 30
patience = 5
best_val_f1 = 0.0
patience_counter = 0

# Cosine Annealing LR Scheduler across 30 epochs
scheduler = CosineAnnealingLR(optimizer, T_max=epochs, eta_min=1e-6)

best_model_path = "/kaggle/working/weights/best_densenet121_full.pth"

print("Starting FarmGuard DenseNet-121 Full-Scale Production Training...")

for epoch in range(epochs):
    model.train()
    running_loss = 0.0
    
    for images, targets in train_loader:
        images, targets = images.to(device), targets.to(device)
        
        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, targets)
        loss.backward()
        optimizer.step()
        
        running_loss += loss.item() * images.size(0)
        
    epoch_train_loss = running_loss / len(train_dataset)
    
    # Validation Pass
    model.eval()
    val_loss = 0.0
    val_preds, val_targets = [], []
    
    with torch.no_grad():
        for images, targets in val_loader:
            images, targets = images.to(device), targets.to(device)
            outputs = model(images)
            loss = criterion(outputs, targets)
            
            val_loss += loss.item() * images.size(0)
            preds = torch.argmax(outputs, dim=1).cpu().numpy()
            val_preds.extend(preds)
            val_targets.extend(targets.cpu().numpy())
            
    epoch_val_loss = val_loss / len(val_dataset)
    
    # Compute Metrics
    acc = accuracy_score(val_targets, val_preds)
    precision = precision_score(val_targets, val_preds, average='macro', zero_division=0)
    recall = recall_score(val_targets, val_preds, average='macro', zero_division=0)
    macro_f1 = f1_score(val_targets, val_preds, average='macro', zero_division=0)
    
    # Step scheduler
    scheduler.step()
    current_lr = scheduler.get_last_lr()[0]
    
    # Log metrics to W&B
    wandb.log({
        "epoch": epoch + 1,
        "train_loss": epoch_train_loss,
        "val_loss": epoch_val_loss,
        "val_accuracy": acc,
        "val_precision": precision,
        "val_recall": recall,
        "val_macro_f1": macro_f1,
        "learning_rate": current_lr
    })
    
    print(f"Epoch [{epoch+1}/{epochs}] | Train Loss: {epoch_train_loss:.4f} | Val Loss: {epoch_val_loss:.4f} | F1: {macro_f1:.4f} | LR: {current_lr:.6f}")
    
    # Checkpoint and Early Stopping logic
    if macro_f1 > best_val_f1:
        best_val_f1 = macro_f1
        patience_counter = 0
        torch.save(model.state_dict(), best_model_path)
        print(f" -> Best production model saved at epoch {epoch+1} with Macro-F1: {best_val_f1:.4f}")
    else:
        patience_counter += 1
        print(f" -> No improvement. Patience counter: {patience_counter}/{patience}")
        
        if patience_counter >= patience:
            print(f"\nEarly stopping triggered! Training successfully halted at epoch {epoch+1}.")
            break

wandb.finish()
print("Training complete! Ready for final test set evaluation.")

Starting FarmGuard DenseNet-121 Full-Scale Production Training...


NameError: name 'train_loader' is not defined

In [6]:
import os

print("Files in /kaggle/working/weights/:")
weights_dir = "/kaggle/working/weights"
if os.path.exists(weights_dir):
    print(os.listdir(weights_dir))
else:
    print("Weights directory does not exist or was wiped due to a kernel restart.")

Files in /kaggle/working/weights/:
Weights directory does not exist or was wiped due to a kernel restart.


In [5]:
import wandb
import torch
import torch.nn as nn
import timm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

# Ensure device is explicitly set if not already defined in your session
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# 1. Initialize final evaluation run on WandB under FarmGuard
wandb.init(project="FarmGuard", name="densenet121-final-test-evaluation")

# 2. Load the best saved production weights from epoch 19
best_model_path = "/kaggle/working/weights/best_densenet121_full.pth"

eval_model = timm.create_model('densenet121', pretrained=False, num_classes=4)
eval_model.load_state_dict(torch.load(best_model_path, map_location=device))
eval_model.to(device)
eval_model.eval()

criterion = nn.CrossEntropyLoss()
test_loss = 0.0
test_preds, test_targets = [], []

print("Running rigorous evaluation on the untouched Test Set...")

# 3. Unseen Test Set Inference Pass
with torch.no_grad():
    for images, targets in test_loader:
        images, targets = images.to(device), targets.to(device)
        outputs = eval_model(images)
        loss = criterion(outputs, targets)
        
        test_loss += loss.item() * images.size(0)
        preds = torch.argmax(outputs, dim=1).cpu().numpy()
        test_preds.extend(preds)
        test_targets.extend(targets.cpu().numpy())

# 4. Calculate Final Empirical Test Metrics
final_test_loss = test_loss / len(test_dataset)
test_acc = accuracy_score(test_targets, test_preds)
test_precision = precision_score(test_targets, test_preds, average='macro', zero_division=0)
test_recall = recall_score(test_targets, test_preds, average='macro', zero_division=0)
test_macro_f1 = f1_score(test_targets, test_preds, average='macro', zero_division=0)

# 5. Print Detailed Reports
print("\n" + "="*45)
print("FARMGUARD: FINAL TEST SET PERFORMANCE REPORT")
print("="*45)
print(f" - Test Loss:         {final_test_loss:.4f}")
print(f" - Test Accuracy:     {test_acc * 100:.2f}%")
print(f" - Test Precision:    {test_precision * 100:.2f}%")
print(f" - Test Recall:       {test_recall * 100:.2f}%")
print(f" - Test Macro-F1:     {test_macro_f1 * 100:.2f}%\n")

print("Class-wise Classification Report:")
print(classification_report(test_targets, test_preds, target_names=class_names))

print("Confusion Matrix:")
print(confusion_matrix(test_targets, test_preds))

# 6. Log final results to W&B
wandb.log({
    "test_loss": final_test_loss,
    "test_accuracy": test_acc,
    "test_precision": test_precision,
    "test_recall": test_recall,
    "test_macro_f1": test_macro_f1
})

wandb.finish()

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/weights/best_densenet121_full.pth'